In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project

Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [2]:
from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline, split_and_save

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install prophet mlflow dagshub -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

In [4]:
import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment("Prophet_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=609c8ef4-e86b-4d2f-a0ea-3dac479fc141&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c8fbf42ae2b9748c9920d278ee887ef1891ae54c75f4622956a905ce9d92e823




Accessing as aochi23

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/7bcf8eafb7104bc699fb04f2b4b2b85b', creation_time=1783520903112, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783520903112, lifecycle_stage='active', name='Prophet_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [5]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)
processed_train, processed_test = split_and_save(full_df)

print(processed_train.shape)
processed_train.head()

(421570, 39)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,lag_52,lag_53,rolling_mean_4,rolling_mean_8,rolling_std_4,rolling_std_8,yoy_growth,dept_avg_sales,store_avg_sales,type_ordinal
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,...,NaN,NaN,32990.77,NaN,12832.106391,NaN,NaN,19213.485088,21710.543621,3


In [ ]:
with mlflow.start_run(run_name="Prophet_Preprocessing"):
    non_sparse = processed_train[processed_train['is_sparse'] == False]
    volume_by_pair = (non_sparse.groupby(['Store', 'Dept'])['Weekly_Sales'].sum().sort_values(ascending=False))
    n_total = len(volume_by_pair)

    np.random.seed(42)
    high_vol = volume_by_pair.iloc[:n_total // 3]
    mid_vol = volume_by_pair.iloc[n_total // 3: 2 * n_total // 3]
    low_vol = volume_by_pair.iloc[2 * n_total // 3:]

    sample_pairs = (list(high_vol.sample(5).index) +
        list(mid_vol.sample(5).index) + list(low_vol.sample(5).index))

    mlflow.log_param("random_seed", 42)
    mlflow.log_param("n_series", len(sample_pairs))
    mlflow.log_param("min_weeks_threshold", 52)
    mlflow.log_param("validation_weeks", 12)
    mlflow.log_param("sampling_strategy", "stratified_by_volume_5_5_5")

    print(f"Selected {len(sample_pairs)} series")
    for s, d in sample_pairs:
        print(f"  Store {s}, Dept {d}")

Selected 15 series
  Store 11, Dept 79
  Store 40, Dept 9
  Store 37, Dept 95
  Store 6, Dept 87
  Store 8, Dept 87
  Store 24, Dept 25
  Store 32, Dept 16
  Store 6, Dept 25
  Store 8, Dept 22
  Store 9, Dept 3
  Store 1, Dept 27
  Store 5, Dept 55
  Store 13, Dept 27
  Store 10, Dept 36
  Store 10, Dept 45
🏃 View run Prophet_Preprocessing at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1/runs/e6764ad999ee47269996c57b5a57cf4d
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1


In [7]:
non_sparse = processed_train[processed_train['is_sparse'] == False]

volume_by_pair = (
    non_sparse
    .groupby(['Store', 'Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)

n_total = len(volume_by_pair)

np.random.seed(42)

high_vol = volume_by_pair.iloc[:n_total // 3]
mid_vol = volume_by_pair.iloc[n_total // 3: 2 * n_total // 3]
low_vol = volume_by_pair.iloc[2 * n_total // 3:]

sample_pairs = (
    list(high_vol.sample(5).index) +
    list(mid_vol.sample(5).index) +
    list(low_vol.sample(5).index)
)

print(f"Selected {len(sample_pairs)} series")
for s, d in sample_pairs:
    print(f"  Store {s}, Dept {d}")

Selected 15 series
  Store 11, Dept 79
  Store 40, Dept 9
  Store 37, Dept 95
  Store 6, Dept 87
  Store 8, Dept 87
  Store 24, Dept 25
  Store 32, Dept 16
  Store 6, Dept 25
  Store 8, Dept 22
  Store 9, Dept 3
  Store 1, Dept 27
  Store 5, Dept 55
  Store 13, Dept 27
  Store 10, Dept 36
  Store 10, Dept 45


In [6]:
def build_walmart_holidays():
    holidays = pd.DataFrame([
        {'holiday': 'SuperBowl',    'ds': '2010-02-12', 'lower_window': 0, 'upper_window': 1},
        {'holiday': 'SuperBowl',    'ds': '2011-02-11', 'lower_window': 0, 'upper_window': 1},
        {'holiday': 'SuperBowl',    'ds': '2012-02-10', 'lower_window': 0, 'upper_window': 1},

        {'holiday': 'LaborDay',     'ds': '2010-09-10', 'lower_window': -1, 'upper_window': 1},
        {'holiday': 'LaborDay',     'ds': '2011-09-09', 'lower_window': -1, 'upper_window': 1},
        {'holiday': 'LaborDay',     'ds': '2012-09-07', 'lower_window': -1, 'upper_window': 1},

        {'holiday': 'Thanksgiving', 'ds': '2010-11-26', 'lower_window': -1, 'upper_window': 2},
        {'holiday': 'Thanksgiving', 'ds': '2011-11-25', 'lower_window': -1, 'upper_window': 2},
        {'holiday': 'Thanksgiving', 'ds': '2012-11-23', 'lower_window': -1, 'upper_window': 2},

        {'holiday': 'Christmas',    'ds': '2010-12-31', 'lower_window': -2, 'upper_window': 1},
        {'holiday': 'Christmas',    'ds': '2011-12-30', 'lower_window': -2, 'upper_window': 1},
        {'holiday': 'Christmas',    'ds': '2012-12-28', 'lower_window': -2, 'upper_window': 1},
    ])
    holidays['ds'] = pd.to_datetime(holidays['ds'])
    return holidays

walmart_holidays = build_walmart_holidays()
print(walmart_holidays)

         holiday         ds  lower_window  upper_window
0      SuperBowl 2010-02-12             0             1
1      SuperBowl 2011-02-11             0             1
2      SuperBowl 2012-02-10             0             1
3       LaborDay 2010-09-10            -1             1
4       LaborDay 2011-09-09            -1             1
5       LaborDay 2012-09-07            -1             1
6   Thanksgiving 2010-11-26            -1             2
7   Thanksgiving 2011-11-25            -1             2
8   Thanksgiving 2012-11-23            -1             2
9      Christmas 2010-12-31            -2             1
10     Christmas 2011-12-30            -2             1
11     Christmas 2012-12-28            -2             1


In [13]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [15]:
def fit_and_evaluate_prophet(series_df, store, dept, holidays, n_test=12,
                               changepoint_prior_scale=0.05, seasonality_mode='additive',
                               yearly_seasonality=True, holidays_mode=None):
    subset = series_df[
        (series_df['Store'] == store) &
        (series_df['Dept'] == dept)
    ].sort_values('Date')

    if len(subset) < n_test + 52 + 10:
        return None

    prophet_df = subset[['Date', 'Weekly_Sales']].rename(
        columns={'Date': 'ds', 'Weekly_Sales': 'y'}
    )

    train_df = prophet_df.iloc[:-n_test]
    val_df = prophet_df.iloc[-n_test:]
    holiday_val = subset['IsHoliday'].iloc[-n_test:].values

    try:
        model = Prophet(
            holidays=holidays,
            yearly_seasonality=yearly_seasonality,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode=seasonality_mode,
            holidays_mode=holidays_mode,
            changepoint_prior_scale=changepoint_prior_scale,
        )
        model.fit(train_df)

        future = model.make_future_dataframe(periods=n_test, freq='W')
        forecast = model.predict(future)

        pred = forecast.tail(n_test)['yhat'].values
        actual = val_df['y'].values

        mae = mean_absolute_error(actual, pred)
        rmse = np.sqrt(np.mean((actual - pred) ** 2))
        w_mae = wmae(actual, pred, holiday_val)

        return {
            'Store': store, 'Dept': dept, 'n_weeks': len(subset),
            'mae': mae, 'rmse': rmse, 'wmae': w_mae,
            'model': model, 'forecast': forecast,
            'train_df': train_df, 'val_df': val_df, 'pred': pred
        }
    except Exception as e:
        print(f"  Failed - Store {store}, Dept {dept}: {e}")
        return None

In [12]:
prophet_configs = [
    {"run_name": "Prophet_Baseline_Additive", "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_CP_0.5",  "changepoint_prior_scale": 0.5,  "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_CP_0.1",  "changepoint_prior_scale": 0.1,  "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_CP_0.05", "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_CP_0.01", "changepoint_prior_scale": 0.01, "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_Fourier_20", "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": 20, "holidays_mode": None},
    {"run_name": "Prophet_Fourier_10", "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": 10, "holidays_mode": None},
    {"run_name": "Prophet_Fourier_5",  "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": 5, "holidays_mode": None},
    {"run_name": "Prophet_Multiplicative", "changepoint_prior_scale": 0.05, "seasonality_mode": "multiplicative", "yearly_seasonality": True, "holidays_mode": None},
    {"run_name": "Prophet_Multiplicative_Holidays", "changepoint_prior_scale": 0.05, "seasonality_mode": "additive", "yearly_seasonality": True, "holidays_mode": "multiplicative"},
]

In [16]:
sweep_results = []

for config in prophet_configs:
    with mlflow.start_run(run_name=config["run_name"]):
        mlflow.log_params(config)

        results_list = []
        for store, dept in sample_pairs:
            result = fit_and_evaluate_prophet(
                processed_train, store, dept, walmart_holidays,
                n_test=12,
                changepoint_prior_scale=config["changepoint_prior_scale"],
                seasonality_mode=config["seasonality_mode"],
                yearly_seasonality=config["yearly_seasonality"],
                holidays_mode=config["holidays_mode"],
            )
            if result is not None:
                results_list.append(result)

        summary_df = pd.DataFrame([{
            'Store': r['Store'], 'Dept': r['Dept'],
            'mae': r['mae'], 'rmse': r['rmse'], 'wmae': r['wmae']
        } for r in results_list])

        mean_mae = summary_df['mae'].mean()
        median_mae = summary_df['mae'].median()
        mean_rmse = summary_df['rmse'].mean()
        mean_wmae = summary_df['wmae'].mean()
        median_wmae = summary_df['wmae'].median()

        mlflow.log_metric("mean_mae", mean_mae)
        mlflow.log_metric("median_mae", median_mae)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("mean_wmae", mean_wmae)
        mlflow.log_metric("median_wmae", median_wmae)
        mlflow.log_metric("n_series_evaluated", len(results_list))

        sweep_results.append({
            "run_name": config["run_name"],
            "mean_mae": mean_mae,
            "mean_wmae": mean_wmae,
            "median_wmae": median_wmae,
            "results_list": results_list,
        })

        print(f"{config['run_name']}: mean_mae={mean_mae:.2f}, mean_wmae={mean_wmae:.2f}")

Prophet_Baseline_Additive: mean_mae=1560.51, mean_wmae=1658.15
🏃 View run Prophet_Baseline_Additive at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1/runs/d0b8caec02db4d7ba9f2f70a7ebd6d9b
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1
Prophet_CP_0.5: mean_mae=1745.21, mean_wmae=1805.54
🏃 View run Prophet_CP_0.5 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1/runs/abff5d39eb6b47119a06bf01e305af09
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1
Prophet_CP_0.1: mean_mae=1578.60, mean_wmae=1644.29
🏃 View run Prophet_CP_0.1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1/runs/c9c90793b240434bb24d618608b8cee8
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1
Prophet_CP_0.05: mean_mae=1560.51, mean_wmae=1658.15
🏃 View run Prophet_CP_0.05 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experi

In [22]:
from tqdm.auto import tqdm

class MultiSeriesProphet(BaseEstimator, RegressorMixin):
    def __init__(self, holidays=None, changepoint_prior_scale=0.05,
                 yearly_seasonality=20, seasonality_mode='additive'):
        self.holidays = holidays
        self.changepoint_prior_scale = changepoint_prior_scale
        self.yearly_seasonality = yearly_seasonality
        self.seasonality_mode = seasonality_mode

    def fit(self, X, y=None):
        self.models_ = {}
        self.skipped_series_ = []
        groups = list(X.groupby(['Store', 'Dept']))

        for (store, dept), grp in tqdm(groups, desc="Fitting Prophet per series"):
            prophet_df = grp[['Date', 'Weekly_Sales']].rename(
                columns={'Date': 'ds', 'Weekly_Sales': 'y'}
            ).sort_values('ds')

            n_valid = prophet_df['y'].notnull().sum()
            if n_valid < 2:
                self.skipped_series_.append((store, dept, n_valid))
                continue

            m = Prophet(
                holidays=self.holidays,
                yearly_seasonality=self.yearly_seasonality,
                weekly_seasonality=False,
                daily_seasonality=False,
                seasonality_mode=self.seasonality_mode,
                changepoint_prior_scale=self.changepoint_prior_scale
            )
            m.fit(prophet_df)
            self.models_[(store, dept)] = m

        self.global_mean_ = X['Weekly_Sales'].mean()

        if self.skipped_series_:
            print(f"Skipped {len(self.skipped_series_)} series with <2 valid observations "
                  f"(will use global mean fallback at predict time).")

        return self

    def predict(self, X):
        preds = pd.Series(index=X.index, dtype=float)

        for (store, dept), grp in X.groupby(['Store', 'Dept']):
            key = (store, dept)
            future = grp[['Date']].rename(columns={'Date': 'ds'})

            if key in self.models_:
                forecast = self.models_[key].predict(future)
                preds.loc[grp.index] = forecast['yhat'].values
            else:
                preds.loc[grp.index] = self.global_mean_

        return preds.values

In [23]:
best_prophet_pipeline = Pipeline(steps=[
    ('prophet', MultiSeriesProphet(
        holidays=walmart_holidays,
        changepoint_prior_scale=0.05,
        yearly_seasonality=20,
        seasonality_mode='additive'
    ))
])

best_prophet_pipeline.fit(processed_train)

Fitting Prophet per series:   0%|          | 0/3331 [00:00<?, ?it/s]

INFO:prophet:n_changepoints greater than number of observations. Using 16.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:n_changepoints greater than number of observations. Using 2.
INFO:prophet:n_changepoints greater than number of observations. Using 2.
INFO:prophet:n_changepoints greater than number of observations. Using 16.
INFO:prophet:n_changepoints greater than number of observations. Using 8.
INFO:prophet:n_changepoints greater than number of observations. Using 0.
INFO:prophet:n_changepoints greater than number of observations. Using 7.
INFO:prophet:n_changepoints greater than number of observations. Using 8.
INFO:prophet:n_changepoints greater than number of observations. Using 5.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 9.
INFO:prophet:n_changepoints greater

Skipped 37 series with <2 valid observations (will use global mean fallback at predict time).


Pipeline(steps=[('prophet',
                 MultiSeriesProphet(holidays=         holiday         ds  lower_window  upper_window
0      SuperBowl 2010-02-12             0             1
1      SuperBowl 2011-02-11             0             1
2      SuperBowl 2012-02-10             0             1
3       LaborDay 2010-09-10            -1             1
4       LaborDay 2011-09-09            -1             1
5       LaborDay 2012-09-07            -1             1
6   Thanksgiving 2010-11-26            -1             2
7   Thanksgiving 2011-11-25            -1             2
8   Thanksgiving 2012-11-23            -1             2
9      Christmas 2010-12-31            -2             1
10     Christmas 2011-12-30            -2             1
11     Christmas 2012-12-28            -2             1))])

In [24]:
fourier20_result = next(
    r for r in sweep_results if r['run_name'] == 'Prophet_Fourier_20'
)

with mlflow.start_run(run_name="Prophet_MultiSeries_Final"):
    mlflow.log_param("changepoint_prior_scale", 0.05)
    mlflow.log_param("yearly_seasonality", 20)
    mlflow.log_param("seasonality_mode", "additive")
    mlflow.log_param("n_series_fit", len(best_prophet_pipeline.named_steps['prophet'].models_))
    mlflow.log_param("n_series_skipped", len(best_prophet_pipeline.named_steps['prophet'].skipped_series_))

    mlflow.log_metric("sweep_mean_mae", fourier20_result['mean_mae'])
    mlflow.log_metric("sweep_mean_wmae", fourier20_result['mean_wmae'])

    mlflow.sklearn.log_model(
        sk_model=best_prophet_pipeline,
        name="prophet_multiseries_pipeline",
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

2026/07/10 13:48:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Prophet_MultiSeries_Final at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1/runs/38e73f3cfbef4a4c8c593126c950ba41
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/1


In [25]:
model_id = "m-b728d29c334b421bbbf9a90a72c752d4"

loaded_pipeline = mlflow.sklearn.load_model(
    f"models:/{model_id}"
)

In [27]:
print(type(loaded_pipeline))
print(loaded_pipeline)

for name, step in loaded_pipeline.named_steps.items():
    print(name)
    print(step)
    print()

from pprint import pprint
pprint(loaded_pipeline.get_params())

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('prophet',
                 MultiSeriesProphet(holidays=         holiday         ds  lower_window  upper_window
0      SuperBowl 2010-02-12             0             1
1      SuperBowl 2011-02-11             0             1
2      SuperBowl 2012-02-10             0             1
3       LaborDay 2010-09-10            -1             1
4       LaborDay 2011-09-09            -1             1
5       LaborDay 2012-09-07            -1             1
6   Thanksgiving 2010-11-26            -1             2
7   Thanksgiving 2011-11-25            -1             2
8   Thanksgiving 2012-11-23            -1             2
9      Christmas 2010-12-31            -2             1
10     Christmas 2011-12-30            -2             1
11     Christmas 2012-12-28            -2             1))])
prophet
MultiSeriesProphet(holidays=         holiday         ds  lower_window  upper_window
0      SuperBowl 2010-02-12             0             1
1      Supe

In [30]:
predictions = loaded_pipeline.predict(processed_test)

print(f"Predictions shape: {predictions.shape}")
print(f"Test set shape: {processed_test.shape}")
print(f"Any NaN predictions: {np.isnan(predictions).sum()}")
print(f"Min: {predictions.min():.2f}, Max: {predictions.max():.2f}, Mean: {predictions.mean():.2f}")

Predictions shape: (115064,)
Test set shape: (115064, 38)
Any NaN predictions: 0
Min: -529775.53, Max: 649267.58, Mean: 16777.35
